<a href="https://colab.research.google.com/github/HimanshiChoubal/CyberShield/blob/main/CyberShield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1 — paste this first, run it, wait for it to finish
!pip install androguard requests python-dotenv groq --quiet
!apt-get install -y default-jdk wget unzip --quiet

# Download and set up apktool
!wget -q https://raw.githubusercontent.com/iBotPeaches/Apktool/master/scripts/linux/apktool -O /usr/local/bin/apktool
!wget -q https://bitbucket.org/iBotPeaches/apktool/downloads/apktool_2.9.3.jar -O /usr/local/bin/apktool.jar
!chmod +x /usr/local/bin/apktool

print("All dependencies installed successfully")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 964.5/964.5 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 71.8 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
unzip is already the newest version (6.0-26ubuntu3.2).
wget is already the newest version (1.21.2-2ubuntu1.1).
The following additional packages will be installed:
  at-spi2-core default-jdk-headless default-jre default-jre-headless
  fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.

In [2]:
# CELL 2 — imports and sanity check
import os
import json
import hashlib
import zipfile
from pathlib import Path
from androguard.misc import AnalyzeAPK
from androguard.core.apk import APK

print("androguard imported successfully")
print("Environment ready")

androguard imported successfully
Environment ready


In [5]:
# CELL 3 — upload an APK file to Colab
from google.colab import files

print("Click 'Choose Files' and select any APK (even a legit one to test)")
uploaded = files.upload()

# Get the filename
apk_path = list(uploaded.keys())[0]
print(f"\nUploaded: {apk_path}")
print(f"File size: {os.path.getsize(apk_path) / 1024:.1f} KB")

Click 'Choose Files' and select any APK (even a legit one to test)


Saving com.amaze.filemanager_122.apk to com.amaze.filemanager_122.apk

Uploaded: com.amaze.filemanager_122.apk
File size: 11180.3 KB


In [6]:
# CELL 4 — compute hashes (used for malware DB lookup later)
def get_file_hashes(filepath):
    hashes = {}
    with open(filepath, 'rb') as f:
        data = f.read()
        hashes['md5']    = hashlib.md5(data).hexdigest()
        hashes['sha1']   = hashlib.sha1(data).hexdigest()
        hashes['sha256'] = hashlib.sha256(data).hexdigest()
    return hashes

hashes = get_file_hashes(apk_path)
print("=== File Fingerprint ===")
for algo, val in hashes.items():
    print(f"  {algo.upper()}: {val}")

=== File Fingerprint ===
  MD5: 2cc0927fd0ff69c45a26c30b56f9812d
  SHA1: 87dbe3f2e4a6520e5d68e3537e7b239c6ef59ee6
  SHA256: aa9b767e00154c2ae3f58fe51585e498dd386e9e7c8a4fec271c089da655c8cc
